# BSM L07G — Pochodzenie aplikacji, zaufanie w runtime, backup i migracja

Pracuj na projekcie `student/apps/lesson_g_app` w Android Studio.

Zaliczanie:
- `G01`: odpowiedz wysyła aplikacja automatycznie.
- `G02-G04`: wypełniasz formularz w notebooku i uruchamiasz komórkę wysyłki.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


# G01 — Manifest i audyt prywatności (auto-submit z aplikacji)

## Cel
Nauczyć się odróżniać:
- deklaracje w `AndroidManifest.xml` (co aplikacja *może* robić),
- realne użycie w kodzie (co aplikacja *robi*),
- prośby runtime (kiedy użytkownik widzi prompt),
- minimalizację (czy da się z mniejszym zakresem uprawnień).

## Część teoretyczna
- Uprawnienie w manifeście nie oznacza automatycznie, że aplikacja dostanie dostęp.
- Dla wielu uprawnień Android wymaga osobnej zgody użytkownika w runtime.
- Dobre praktyki:
  1. prosić o uprawnienie dopiero gdy funkcja jest potrzebna,
  2. uzasadniać w UI, po co to jest,
  3. mieć fallback (aplikacja nie powinna się „wywracać” po odmowie).

## Co masz zrobić (krok po kroku)
1. Otwórz `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik: `app/src/main/AndroidManifest.xml`.
1. Przejrzyj wszystkie wpisy `<uses-permission ...>`.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Znajdź w UI sekcję „Student / Task 1” i zobacz, jakie uprawnienia aplikacja żąda po kliknięciu „Request permissions”.
1. Zrób mapę: każde uprawnienie -> jaka funkcja je wykorzystuje (mapa, zdjęcia, kamera, internet).
1. Sprawdź zachowanie fallback:
- co aplikacja pokazuje, gdy nie ma lokalizacji,
- co pokazuje, gdy brak dostępu do zdjęć,
- co pokazuje, gdy nie ma kamery.

## Jak zaliczasz (auto-submit)
1. Uruchom aplikację.
1. Wpisz swoje `Student ID`.
1. Kliknij „Request permissions” i przejdź cały przepływ.
1. Gdy ID + wymagane uprawnienia są OK, aplikacja sama wyśle odpowiedź dla `G01`.


# G02 — APK / bundle provenance check

## O co chodzi
W tym ćwiczeniu nie sprawdzasz, czy aplikacja ma poprawną nazwę paczki. To za mało. Chodzi o coś konkretniejszego: czy uruchomiony build jest podpisany oczekiwaną tożsamością i czy nie wygląda na przepakowany albo podmieniony.

Musisz rozróżnić dwa poziomy zaufania:
- **install-time trust**: Android sprawdza podpis przy instalacji i aktualizacji.
- **runtime trust**: aplikacja sama podejmuje decyzję, czy ufa aktualnie uruchomionemu buildowi i czy pozwala wykonać wrażliwą operację.

W praktyce student ma zbudować prosty, testowalny mechanizm, który:
- pobiera informacje o podpisie aplikacji z systemu,
- porównuje je z oczekiwaną tożsamością,
- odrzuca build, który nie pasuje,
- nie myli `packageName` z rzeczywistą tożsamością builda.

## Co masz otworzyć
1. Otwórz `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
2. Znajdź `ProvenanceState` i `TaskCompletion.task2Check(...)`.
3. Otwórz `app/src/test/java/com/example/secretlab/lab/TaskCompletionStudentTest.kt`.
4. Przeczytaj test `task2CodeAppearsOnlyWhenProvenanceChecksPass` i wypisz sobie trzy warunki, które muszą być spełnione jednocześnie.
5. Otwórz `app/src/main/java/com/example/secretlab/MainActivity.kt`.
6. Zobacz, że starter ma już importy `PackageManager` i `SigningInfo`. To jest trop, który prowadzi do implementacji provenance check.

## Co masz zrobić krok po kroku
1. Dodaj albo uzupełnij pomocniczy kod, który pobiera informacje o podpisie aktualnej aplikacji.
2. Użyj `PackageManager.getPackageInfo(...)` z flagą `PackageManager.GET_SIGNING_CERTIFICATES`.
3. Z odczytanego `PackageInfo` przejdź do `signingInfo`.
4. Z `SigningInfo` pobierz podpisy APK przez `getApkContentsSigners()`.
5. Dla podpisu, który traktujesz jako aktywny podpis builda, pobierz bajty przez `Signature.toByteArray()`.
6. Policz z tych bajtów skrót SHA-256 przez `MessageDigest`.
7. Porównaj wynik z wartością oczekiwaną przez lab.
8. Na tej podstawie ustaw `signingIdentityMatchesExpected`.
9. Zdecyduj, kiedy ustawiasz `buildLooksTampered`.
   W tym labie nie chodzi o wykrywanie wszystkich możliwych manipulacji, tylko o sensowną politykę odrzucenia builda, który nie zgadza się z oczekiwanym podpisem albo ma niespójny stan danych o podpisie.
10. Ustaw `installTimeTrustIsSeparatedFromRuntimeTrust` dopiero wtedy, gdy w kodzie rzeczywiście rozdzielasz te dwa etapy.
    To pole nie ma znaczyć „Android coś sprawdził przy instalacji”, tylko „moja logika runtime nie ufa ślepo temu, że aplikacja się uruchomiła”.
11. Dopiero po poprawnym zbudowaniu `ProvenanceState` spraw, żeby `task2Check(...)` zwracało prawdę dla poprawnego przypadku i fałsz dla pozostałych.

## Na co uważać
- Nie opieraj decyzji tylko na `packageName`.
- Nie wpisuj odpowiedzi „na sztywno” do `task2Check(...)` bez realnej logiki.
- Nie mieszaj warunku „czy podpis pasuje” z warunkiem „czy aplikacja się uruchomiła”.
- Test ma być konsekwencją Twojej logiki, a nie odwrotnie.

## Co kliknąć, żeby sprawdzić wynik
1. Android Studio -> `View` -> `Tool Windows` -> `Gradle`.
2. W drzewie projektu przejdź do: `lesson_g_app` -> `app` -> `Tasks` -> `verification`.
3. Uruchom `testDebugUnitTest`.
4. Potem w terminalu, w katalogu `student/apps/lesson_g_app`, uruchom:
   `./gradlew :app:bsmEvidence`
5. Jeżeli wszystko jest poprawne, w konsoli pojawi się 5-znakowy kod dla `G02`.

## Dokumentacja do tego konkretnego zadania
- `PackageManager`: https://developer.android.com/reference/android/content/pm/PackageManager
- `PackageManager.getPackageInfo(...)`: https://developer.android.com/reference/android/content/pm/PackageManager#getPackageInfo(java.lang.String,int)
- `PackageInfo.signingInfo`: https://developer.android.com/reference/android/content/pm/PackageInfo#signingInfo
- `SigningInfo`: https://developer.android.com/reference/android/content/pm/SigningInfo
- `SigningInfo.getApkContentsSigners()`: https://developer.android.com/reference/android/content/pm/SigningInfo#getApkContentsSigners()
- `Signature.toByteArray()`: https://developer.android.com/reference/android/content/pm/Signature#toByteArray()
- `MessageDigest`: https://developer.android.com/reference/java/security/MessageDigest

## Co wpisać do `final_answer`
Wpisz tylko 5-znakowy kod wypisany przez `:app:bsmEvidence`.


In [ ]:
#@title G02 — Formularz odpowiedzi
code_g02 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g02.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G02", final_answer)


# G03 — Integrity-gated backend request

## O co chodzi
Po G02 wiesz już, czy build wygląda na ten, któremu chcesz ufać. To nadal nie wystarcza, żeby backend miał przyjąć żądanie bez dodatkowych warunków.

W tym ćwiczeniu musisz zrobić bramkę zaufania dla requestu. Request ma przejść tylko wtedy, gdy:
- verdict jest poprawny,
- tożsamość aplikacji jest zgodna,
- request jest związany z tą tożsamością, a nie tylko wysłany z „jakiegoś” klienta.

W starterze ten model jest opisany przez `IntegrityState` i `TaskCompletion.task3Check(...)`.

## Co masz otworzyć
1. Otwórz `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
2. Znajdź `IntegrityState` i `task3Check(...)`.
3. Otwórz `app/src/test/java/com/example/secretlab/lab/TaskCompletionStudentTest.kt`.
4. Przeczytaj test `task3CodeAppearsOnlyWhenIntegrityVerdictAndBindingAreReady`.
5. Otwórz `app/src/main/java/com/example/secretlab/MainActivity.kt`.
6. Znajdź funkcje `submitTask1(...)` i `submitAnswer(...)`.
7. Sprawdź, jak budowany jest payload JSON i w którym miejscu najłatwiej zatrzymać wysyłkę, gdy warunki zaufania nie są spełnione.

## Co masz zrobić krok po kroku
1. Zaimplementuj logikę, która buduje `IntegrityState`.
2. Ustal, skąd bierzesz `verdict`.
   W tym labie to może być prosty, testowalny model, ale ma reprezentować decyzję „pozwól / nie pozwól” po wykonaniu sprawdzeń.
3. Ustal, jak potwierdzasz `appPackageNameMatches`.
   To pole samo w sobie nie wystarcza do zaufania, ale ma być jednym z warunków bramki.
4. Dodaj binding requestu do tożsamości aplikacji.
   Oznacza to, że request nie powinien być akceptowany tylko dlatego, że ma prawidłowy `taskId` i `studentId`.
   Musisz dołączyć albo wyliczyć taką informację, która łączy żądanie z aktualną, zaufaną tożsamością builda.
5. Przed wysyłką wywołaj logikę sprawdzającą `task3Check(...)`.
6. Jeżeli wynik jest negatywny:
- nie wysyłaj requestu,
- ustaw czytelny komunikat w UI,
- nie próbuj „ratować” zadania niejawną wysyłką rezerwową.
7. Jeżeli wynik jest pozytywny, dopiero wtedy pozwól na request.

## Na co uważać
- Nie sprowadzaj zadania do jednego `if (verdict == "ALLOW")`.
- Nie zostawiaj sytuacji, w której request idzie dalej mimo niespełnionego bindingu.
- Nie mieszaj G03 z G01. Auto-submit dla Task 1 ma zostać osobną ścieżką.

## Co kliknąć, żeby sprawdzić wynik
1. Android Studio -> `View` -> `Tool Windows` -> `Gradle`.
2. Uruchom `testDebugUnitTest` w `lesson_g_app -> app -> Tasks -> verification`.
3. Potem w terminalu w katalogu `student/apps/lesson_g_app` uruchom:
   `./gradlew :app:bsmEvidence`
4. Jeżeli logika jest poprawna, pojawi się 5-znakowy kod dla `G03`.

## Dokumentacja do tego konkretnego zadania
- `HttpURLConnection`: https://developer.android.com/reference/java/net/HttpURLConnection
- `URL`: https://developer.android.com/reference/java/net/URL
- `URL.openConnection()`: https://developer.android.com/reference/java/net/URL#openConnection()

## Co wpisać do `final_answer`
Wpisz tylko 5-znakowy kod wypisany przez `:app:bsmEvidence`.


In [ ]:
#@title G03 — Formularz odpowiedzi
code_g03 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g03.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G03", final_answer)


# G04 — Backup, migracja i higiena sekretów

## O co chodzi
To zadanie nie polega tylko na odczytaniu 5-znakowego sekretu. Chodzi o zrozumienie pełnej ścieżki, jaką w tym starterze przechodzą dane wrażliwe, oraz o ocenę, które elementy tej ścieżki mogą podlegać backupowi i migracji, a które nie powinny.

Aktualny starter ma dwa różne modele pracy z sekretami:
- model **build-time**: sekret trafia do aplikacji przez `local.properties` -> `build.gradle.kts` -> `BuildConfig` -> `AppSecrets` -> natywny helper w `secret_keys.cpp`,
- model **runtime/local storage**: sekret jest przechowywany lokalnie przez `AppSecretsStore` i `SecurePrefs`.

To rozróżnienie jest w tym ćwiczeniu najważniejsze. Masz umieć odpowiedzieć:
- które dane są wstrzykiwane do builda,
- które dane są zapisywane już po uruchomieniu aplikacji,
- które z nich mogą zostać przeniesione między urządzeniami przez backup,
- które powinny zostać z backupu wykluczone.

## Co masz otworzyć
1. Otwórz `student/apps/lesson_g_app/local.properties`.
2. Znajdź `task4_secret_b64`.
3. Zobacz, że `map_api_key_b64` nie jest jeszcze ustawione.
4. Otwórz `app/build.gradle.kts`.
5. Znajdź `buildConfigField(...)` dla `MAP_API_KEY_B64` i `TASK4_SECRET_B64`.
6. Otwórz `app/src/main/java/com/example/secretlab/secure/AppSecrets.kt`.
7. Zobacz, że aplikacja nie wywołuje już `decodeB64(...)`, tylko `decryptBlob(...)`.
8. Otwórz `app/src/main/cpp/secret_keys.cpp`.
9. Sprawdź, co dokładnie dzieje się w `Java_com_example_secretlab_secure_AppSecrets_decryptBlob(...)`.
10. Przejdź krok po kroku przez tę funkcję:
- najpierw wejściowy string jest dekodowany z Base64,
- potem wynik przechodzi przez `xxteaDecrypt(...)`,
- dopiero końcowy wynik wraca do aplikacji jako plaintext.
11. Otwórz `app/src/main/java/com/example/secretlab/secure/AppSecretsStore.kt`.
12. Porównaj ten plik z `AppSecrets.kt`.
    `AppSecrets.kt` czyta sekrety dostarczone na etapie builda, a `AppSecretsStore.kt` operuje na sekretach zapisanych lokalnie przez aplikację.
13. Otwórz `app/src/main/AndroidManifest.xml`.
14. Sprawdź `android:allowBackup="true"`.
15. Otwórz `app/src/main/res/xml/backup_rules.xml`.
16. Zobacz, że obecne reguły backupu są puste, więc aplikacja nie ma jeszcze sensownej polityki dla danych wrażliwych.

## Co masz zrobić krok po kroku
1. Ustal, które dane w projekcie należą do ścieżki build-time secrets.
   W aktualnym starterze są to wartości `MAP_API_KEY_B64` i `TASK4_SECRET_B64`, ładowane z `local.properties` do `BuildConfig`, a potem odszyfrowywane przez `AppSecrets` i natywny helper.
2. Ustal, które dane należą do ścieżki runtime/local storage.
   W aktualnym projekcie odpowiada za to `AppSecretsStore` i warstwa `SecurePrefs`.
3. Rozdziel w analizie te dwa przypadki.
   Sekret dostarczany na etapie builda nie jest "przywracany z backupu" w taki sam sposób jak sekret zapisany przez aplikację po instalacji.
4. Sprawdź, skąd wziąć klucz API mapy.
   Otwórz `app/src/main/java/com/example/secretlab/MainActivity.kt` i znajdź `ApiMapCard(...)` oraz `buildStaticMapUrl(...)`.
   Zobaczysz tam, że aplikacja buduje URL do statycznej mapy Mapbox z parametrem `access_token`, więc potrzebny jest token Mapbox.
5. Dodaj własny token Mapbox do `local.properties` jako `map_api_key_b64`.
   Uwaga: nazwa kończy się na `_b64`, ale to jest tylko warstwa transportowa do przekazania blobu przez konfigurację Gradle. Ochrona w tym starterze nie polega wyłącznie na Base64.
6. Oceń, czy aktualna polityka backupu jest wystarczająca.
   Przy `android:allowBackup="true"` i pustym `backup_rules.xml` odpowiedź powinna być krytyczna.
7. Zaproponuj i wdroż sensowniejszą politykę:
- albo wyłącz backup całkowicie,
- albo zostaw backup, ale jawnie wyklucz dane lokalne, które nie powinny migrować.
8. Upewnij się, że nie mieszasz sekretu build-time z sekretem runtime.
   To, że coś jest ukryte w natywnym helperze, nie oznacza automatycznie, że problem backupu lokalnych danych jest rozwiązany.

## Dodatkowa lektura pomocnicza
Możesz wykorzystać jako materiał pomocniczy:
- https://al-e-shevelev.medium.com/a-secure-way-to-store-api-keys-in-android-applications-238135709067
- https://docs.mapbox.com/help/glossary/access-token/

Czytaj to krytycznie. W tym starterze nie chodzi o stwierdzenie „sekret jest bezpieczny, bo jest w C++”. Chodzi o rozumienie całej ścieżki: skąd sekret trafia do builda, jak jest odszyfrowywany i które dane aplikacji mogą zostać przeniesione przez backup.

## Dokumentacja do tego konkretnego zadania
- `android:allowBackup`: https://developer.android.com/guide/topics/manifest/application-element#allowbackup
- `android:fullBackupContent`: https://developer.android.com/guide/topics/manifest/application-element#fullBackupContent
- `BuildConfig`: https://developer.android.com/build/gradle-tips#share-custom-fields-and-resource-values-with-your-apps-code
- `System.loadLibrary(...)`: https://developer.android.com/reference/java/lang/System#loadLibrary(java.lang.String)

## Jak odczytać odpowiedź
1. Nie wpisuj wartości „z głowy”.
2. Przejdź ścieżkę sekretu Task 4:
   `local.properties` -> `build.gradle.kts` -> `BuildConfig.TASK4_SECRET_B64` -> `AppSecrets.readTask4Secret()` -> `decryptBlob(...)` w `secret_keys.cpp`.
3. Odczytaj końcową, odszyfrowaną 5-znakową wartość.
4. To jest odpowiedź do `G04`.

## Co wpisać do `final_answer`
Wpisz tylko 5-znakową wartość sekretu dla `G04`.


In [ ]:
#@title G04 — Formularz odpowiedzi
secret_g04 = ""  #@param {type:"string"}

final_answer = prepare_answer(secret_g04.strip())
print(final_answer)


In [ ]:
zapisz_i_wyslij("G04", final_answer)
